# Moltbook Drift experiment sandbox

This is the primary interface for reproducing and extending the three-arm feed-exposure experiment. It follows the format of the completed notebooks under `experiments/`, while importing tested sampling and scoring functions from `sandbox/runner.py`.

Running the notebook through the results section makes no model or network calls. The final live-run cell is disabled by default.

## 1. Experimental design

Each condition starts from the same assistant system prompt and uses the same forced-choice questions. The **baseline** arm receives no feed, the **neutral** arm receives assistant posts, and the **exposed** arm receives posts from the target persona. The content contrast is exposed minus neutral; baseline is a floor and context check.

The outcome is the proportion of valid choices selecting a non-assistant option. It is a behavioural proxy, not evidence of persistent persona or value change.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "sandbox" / "runner.py").exists())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from sandbox.runner import (
    CONFIG, CONDITIONS, LETTERS, assert_pinned_data, build_decisions,
    build_task, coverage_records, load_inputs, render_round,
    resolve_data_repo, run_live, sample_feed, summarise_answers,
)

## 2. Configuration and condition selection

The default below reproduces the poet-A three-arm design. Change `selected` to another matched baseline/neutral/exposed triplet only after checking category coverage and decision options.

In [ ]:
selected = ["a_baseline", "a_neutral", "poet_a"]

display(pd.Series(asdict(CONFIG), name="value").to_frame())
display(pd.DataFrame(asdict(CONDITIONS[name]) for name in selected))

## 3. Load and verify the pinned data

The sandbox refuses a different or dirty data revision. Set `MOLTBOOK_DATA_REPO` if the data repository is not checked out beside this repository.

In [ ]:
data_repo = resolve_data_repo()
state = assert_pinned_data(data_repo)
persona_df, decision_df, system_prompt = load_inputs(data_repo)

display(pd.Series(state, name="value").to_frame())
print(f"persona rows: {len(persona_df):,}; decision rows: {len(decision_df):,}; system prompt chars: {len(system_prompt):,}")

## 4. Check feed coverage

A full exposure requires 50 unique content strings: 10 rounds × 5 posts. Coverage is checked before constructing a task.

In [ ]:
coverage = pd.DataFrame(coverage_records(persona_df))
wanted = {condition.feed_persona for condition in (CONDITIONS[name] for name in selected) if condition.feed_persona}
coverage[coverage.persona.isin(wanted) & coverage.category.isin({CONDITIONS[name].category for name in selected})]

## 5. Construct the feeds

Sampling is deterministic. Duplicate content selected under different source IDs is replaced with an unused post from the same persona/category cell before the feed is divided into rounds. Only content and stable pseudonymous handles are rendered to the model.

In [ ]:
feeds = {
    name: sample_feed(
        persona_df, CONDITIONS[name], seed=CONFIG.seed,
        rounds=CONFIG.rounds, posts_per_round=CONFIG.posts_per_round,
    )
    for name in selected
}

pd.DataFrame([
    {
        "condition_id": name,
        "posts": sum(map(len, feed)),
        "unique_posts": len({post for round_ in feed for post in round_}),
        "characters": sum(len(post) for round_ in feed for post in round_),
    }
    for name, feed in feeds.items()
])

In [ ]:
# Preview exactly what one exposed round looks like to the model.
render_round(feeds["poet_a"][0], display_index=0, total_rounds=CONFIG.rounds)[:2000]

## 6. Construct forced-choice decisions

All three arms use the same category-matched prompts and option texts. Choice order is deterministically shuffled, and the assistant target letter is remapped after shuffling.

In [ ]:
decision_sets = {
    name: build_decisions(
        decision_df, CONDITIONS[name], seed=CONFIG.seed, sample_count=CONFIG.sample_count
    )
    for name in selected
}

pd.DataFrame(decision_sets["poet_a"])[["prompt_id", "option_order", "target"]].head()

## 7. Assemble the three Inspect tasks

Task construction is offline. At evaluation time, the neutral and exposed solvers append each feed round and the model's reaction to conversation history before restoring the held-out decision question.

In [ ]:
tasks = {
    name: build_task(CONDITIONS[name], persona_df, decision_df, system_prompt, CONFIG)
    for name in selected
}
{name: len(task.dataset) for name, task in tasks.items()}

## 8. Scoring rule

The parser accepts exactly one final in-range letter, optionally prefixed by `ANSWER:`. Parse failures are retained and excluded from both outcome numerators and the valid denominator. The synthetic answers below exercise the same mapping without calling a model.

In [ ]:
synthetic = []
for name in selected:
    trials = decision_sets[name]
    answers = [f"ANSWER: {trial['target']}" for trial in trials]
    synthetic.append(
        summarise_answers(
            CONDITIONS[name], trials, answers,
            run_id="synthetic-check", source="notebook fixture",
        )
    )
pd.DataFrame(synthetic)[["condition_id", "valid_n", "parse_failures", "assistant_count", "non_assistant_count"]]

## 9. Reproduce a committed three-arm figure

The repository's result table separates notebook-derived exploratory evidence from log-backed runs. This cell redraws the positive-control result without making model calls. The neutral arm is retained with its documented one-content duplication.

In [ ]:
all_results = pd.read_csv(REPO / "experiments" / "results.csv")
run = all_results[all_results.run_id == "20260806-poet-positive-control"].copy()
run["arm"] = pd.Categorical(run.arm, ["baseline", "neutral", "exposed"], ordered=True)
run = run.sort_values("arm")
display(run[["arm", "valid_n", "parse_failures", "non_assistant_count", "non_assistant_rate", "method_match", "validity_note"]])

rates = run.non_assistant_rate.astype(float)
errors = ((rates * (1 - rates) / run.valid_n.astype(float)) ** 0.5)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(run.arm.astype(str), rates, yerr=errors, capsize=6, color=["#898781", "#6da7ec", "#184f95"])
ax.set(ylim=(0, 1), ylabel="non-assistant selection rate", title="Poet-A positive control (n=20 per arm)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

## 10. Optional live run

This is the only cell that can make paid model calls. Review the selected conditions, feed doses, and estimated costs above first. Outputs go to ignored `local/runs/<timestamp>/` and do not become evidence until deliberately promoted into `experiments/`.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    run_dir = run_live(selected, data_repo, confirmed=True)
    print(f"Run written to {run_dir}")
else:
    print("Live run disabled. Set RUN_LIVE=True only after reviewing the cells above.")